# torchtune
pytorch中提供的LLM工具

In [5]:
!pip install torch torchvision torchaudio

  Using cached torchvision-0.17.2-cp310-cp310-macosx_11_0_arm64.whl.metadata (6.6 kB)
  Using cached torchaudio-2.2.2-cp310-cp310-macosx_11_0_arm64.whl.metadata (6.4 kB)
  Using cached pillow-10.3.0-cp310-cp310-macosx_11_0_arm64.whl.metadata (9.2 kB)
Using cached torchvision-0.17.2-cp310-cp310-macosx_11_0_arm64.whl (1.6 MB)
Using cached torchaudio-2.2.2-cp310-cp310-macosx_11_0_arm64.whl (1.8 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 2.5 MB/s eta 0:00:0000:0100:010m


In [1]:
!pip install torchtune

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 37.0 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 31.7 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 17.7 kB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.1/192.1 kB 30.4 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.3/54.3 kB 25.3 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 31.5 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 388.9/388.9 kB 41.0 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 41.0 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 33.2 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.9/410.9 kB 34.9 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 38.9 kB/s 

In [6]:
# 查看torchtune是否安装成功
!tune

[2024-04-17 17:30:13,532] torch.distributed.elastic.multiprocessing.redirects: [WARNING] NOTE: Redirects are currently not supported in Windows or MacOs.
usage: tune [-h] {download,ls,cp,run,validate} ...

Welcome to the TorchTune CLI!

options:
  -h, --help            show this help message and exit

subcommands:
  {download,ls,cp,run,validate}
    download            Download a model from the Hugging Face Hub.
    ls                  List all built-in recipes and configs
    cp                  Copy a built-in recipe or config to a local path.
    run                 Run a recipe. For distributed recipes, this supports
                        all torchrun arguments.
    validate            Validate a config and ensure that it is well-formed.


## Fine-Tune LLM

### 1. 下载模型

In [ ]:
# 通过Huggingface进行身份验证下载模型
!tune download meta-llama/Llama-2-7b-hf \
  --output-dir /tmp/Llama-2-7b-hf \
  --hf-token <ACCESS TOKEN>

### 2. 选择配置

- 配置用于与 LLM 交互，包括训练、 推理、评估和量化。
- 配置有三个部分组成
    - 可配置参数，通过 yaml 配置和命令行覆盖指定
    - 配方脚本，将所有内容放在一起的入口点，包括解析和验证配置、设置环境以及正确使用配方类
    - 配方类，训练所需的核心逻辑，通过一组 API 公开

In [10]:
!tune ls

[2024-04-17 18:04:16,450] torch.distributed.elastic.multiprocessing.redirects: [WARNING] NOTE: Redirects are currently not supported in Windows or MacOs.
RECIPE                                   CONFIG                                  
full_finetune_single_device              llama2/7B_full_low_memory               
                                         mistral/7B_full_low_memory              
full_finetune_distributed                llama2/7B_full                          
                                         llama2/13B_full                         
                                         mistral/7B_full                         
                                         gemma/2B_full                           
lora_finetune_single_device              llama2/7B_lora_single_device            
                                         llama2/7B_qlora_single_device           
                                         mistral/7B_lora_single_device           
                          

以下是在一个设备上使用LoRA微调LLaMA2

### 3. 微调模型

#### 修改配置
YAML 配置包含运行配方所需的大多数重要信息。 您可以设置超参数、指定 WandB 等指标记录器、选择新数据集等。 有关当前支持的所有数据集的列表，请参阅 torchtune.datasets。

方式一：从命令行覆盖现有参数

In [ ]:
!tune run <RECIPE> --config <CONFIG> epochs=1

方式二：通过“tune cp”复制配置并直接修改

In [ ]:
!tune cp llama2/7B_lora_single_device custom_config.yaml

#### 训练模型

In [ ]:
!tune run lora_finetune_single_device --config llama2/7B_lora_single_device epochs=1